# Trabajo Práctico N°2 — Entrenamiento y Optimización del Modelo

**Alumno:** Gonzalo Zarazaga

---

Este notebook entrena un modelo **XGBoost** para predecir si una persona es fumadora o no (`smoking`)
y optimiza sus hiperparámetros para reducir el overfitting detectado en el baseline.

**Métrica de evaluación:** F1-Score para la clase positiva (`smoking = 1`).

**Insumo:** `data/processed/smoking_train_processed.csv`  
**Salidas:** `models/xgboost_baseline.joblib` · `models/xgboost_optimized.joblib`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score
)
from xgboost import XGBClassifier

PATH_PROC   = Path("../data/processed")
PATH_MODELS = Path("../models")

RANDOM_STATE = 42
TEST_SIZE    = 0.2

## 1. Carga del dataset procesado

In [ ]:
df = pd.read_csv(PATH_PROC / "smoking_train_processed.csv")

print(f"Shape: {df.shape}")
print(f"Nulls: {df.isna().sum().sum()}")
print(f"Object cols: {list(df.select_dtypes('object').columns)} (debe ser vacío)")
df.head(3)

## 2. División train / test

Se divide el dataset etiquetado en:
- **Train (80%):** conjunto que usa el modelo para aprender
- **Test (20%):** conjunto que simula datos no vistos — se usa solo para evaluación final

Buenas prácticas aplicadas:
- `random_state=42` → aleatoriedad reproducible
- `stratify=y` → garantiza que la proporción de fumadores sea la misma en train y test
- El conjunto de test **no se toca** hasta la sección de evaluación

In [ ]:
X = df.drop(columns=["smoking"])
y = df["smoking"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   |  y_test:  {y_test.shape}")
print()

prop_total = y.mean()
prop_train = y_train.mean()
prop_test  = y_test.mean()

print("Proporción de fumadores (clase 1):")
print(f"  Total:  {prop_total:.4f}  ({prop_total*100:.1f}%)")
print(f"  Train:  {prop_train:.4f}  ({prop_train*100:.1f}%)")
print(f"  Test:   {prop_test:.4f}  ({prop_test*100:.1f}%)")
print(f"\nFeatures ({len(X.columns)}): {list(X.columns)}")

## 3. Entrenamiento del modelo baseline

Se entrena un modelo **XGBoost** con parámetros de referencia para establecer un punto de partida.

### Por qué XGBoost es adecuado para este problema
- No requiere escalado de variables
- Es robusto a outliers
- La distribución de las features no afecta su rendimiento
- Maneja internamente la multicolinealidad

### Parámetros del baseline

| Parámetro | Valor | Efecto |
|---|---|---|
| `n_estimators` | 100 | Número de árboles |
| `max_depth` | 6 | Profundidad máxima de cada árbol |
| `learning_rate` | 0.1 | Tasa de aprendizaje |
| `subsample` | 0.8 | Fracción de filas por árbol — regularización |
| `colsample_bytree` | 0.8 | Fracción de columnas por árbol — regularización |
| `random_state` | 42 | Reproducibilidad |

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    verbosity=0
)

model.fit(X_train, y_train)
print("Modelo baseline entrenado.")

## 4. Evaluación baseline: train vs test

La comparación train/test permite detectar **overfitting** (el modelo memorizó el train pero no generaliza)
o **underfitting** (el modelo no aprendió lo suficiente).

**Métrica principal:** F1-Score para la clase `1` (fumador).

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

In [ ]:
print("=" * 45)
print("  REPORTE DE CLASIFICACIÓN — TRAIN")
print("=" * 45)
print(classification_report(
    y_train, y_train_pred,
    target_names=["No fumador (0)", "Fumador (1)"]
))

In [ ]:
print("=" * 45)
print("  REPORTE DE CLASIFICACIÓN — TEST")
print("=" * 45)
print(classification_report(
    y_test, y_test_pred,
    target_names=["No fumador (0)", "Fumador (1)"]
))

In [ ]:
f1_train = f1_score(y_train, y_train_pred)
f1_test  = f1_score(y_test,  y_test_pred)
gap      = f1_train - f1_test

print("--- Métrica principal: F1-Score (clase 1 = fumador) ---")
print(f"  Train:  {f1_train:.4f}")
print(f"  Test:   {f1_test:.4f}")
print(f"  Gap:    {gap:+.4f}")
print()

if gap > 0.05:
    print("⚠  Gap > 5 pp → overfitting detectado. Se procede a optimizar.")
elif gap < -0.02:
    print("⚠  Test > Train → comportamiento inusual. Revisar el split.")
else:
    print("✓  Gap dentro de rangos aceptables.")

## 5. Matrices de confusión (baseline)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (y_true, y_pred, titulo) in zip(axes, [
    (y_train, y_train_pred, "Train"),
    (y_test,  y_test_pred,  "Test")
]):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No fumador (0)", "Fumador (1)"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Matriz de Confusión — {titulo} (baseline)",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicción", fontsize=10)
    ax.set_ylabel("Real", fontsize=10)

plt.tight_layout()
plt.show()

## 6. Importancia de features (baseline)

XGBoost calcula la importancia de cada variable según cuánto contribuye a mejorar
las predicciones a lo largo de todos los árboles. Sirve para confirmar los hallazgos del EDA.

In [ ]:
importancias = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))

colores = [
    "#e74c3c" if imp == importancias.max() else
    "#3498db" if imp >= importancias.quantile(0.75) else
    "#95a5a6"
    for imp in importancias.values
]

ax.barh(importancias.index, importancias.values,
        color=colores, edgecolor="white", height=0.75)

for i, (val, name) in enumerate(zip(importancias.values, importancias.index)):
    ax.text(val + 0.001, i, f"{val:.4f}", va="center", ha="left", fontsize=8)

ax.set_xlabel("Importancia (gain normalizado)", fontsize=11)
ax.set_title(
    "Importancia de Features — XGBoost Baseline\n"
    "Rojo = mayor · Azul = cuartil superior · Gris = resto",
    fontsize=12, fontweight="bold"
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 5 más importantes:")
print(importancias.sort_values(ascending=False).head(5).to_string())
print("\nTop 5 menos importantes:")
print(importancias.sort_values(ascending=True).head(5).to_string())

## 7. Guardado del modelo baseline

In [ ]:
out_baseline = PATH_MODELS / "xgboost_baseline.joblib"
joblib.dump(model, out_baseline)
print(f"✓ Baseline guardado: {out_baseline}  ({out_baseline.stat().st_size/1024:.1f} KB)")

---
## 8. Análisis de overfitting: efecto de `n_estimators`

Antes de lanzar una búsqueda de hiperparámetros completa, se analiza cómo evoluciona el F1-Score en
train y test al variar el número de árboles. Esto permite identificar visualmente en qué rango
el modelo empieza a memorizar el conjunto de entrenamiento.

> **Principio:** cuanto más árboles, más complejo el modelo y más riesgo de overfitting.
> El objetivo es encontrar el punto donde el F1-test se estabiliza antes de que el gap empiece a crecer.

In [ ]:
n_valores = [10, 25, 50, 75, 100, 150, 200, 300]
resultados = []

print(f"{'n_estimators':>14}  {'F1-train':>9}  {'F1-test':>8}  {'gap':>8}")
print("-" * 45)

for n in n_valores:
    m = XGBClassifier(
        n_estimators=n, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0
    )
    m.fit(X_train, y_train)
    f1_tr = f1_score(y_train, m.predict(X_train))
    f1_te = f1_score(y_test,  m.predict(X_test))
    resultados.append({"n_estimators": n, "f1_train": f1_tr, "f1_test": f1_te, "gap": f1_tr - f1_te})
    print(f"{n:>14}  {f1_tr:>9.4f}  {f1_te:>8.4f}  {f1_tr-f1_te:>+8.4f}")

df_curva = pd.DataFrame(resultados)
mejor_n  = int(df_curva.loc[df_curva.f1_test.idxmax(), "n_estimators"])
print(f"\nMejor n_estimators por F1-test: {mejor_n}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel izquierdo: curvas F1 ---
axes[0].plot(df_curva.n_estimators, df_curva.f1_train, "o-", color="#3498db", label="Train")
axes[0].plot(df_curva.n_estimators, df_curva.f1_test,  "o-", color="#e74c3c", label="Test")
axes[0].axvline(x=100, color="gray", linestyle="--", linewidth=1.2,
                alpha=0.7, label="Baseline (100)")
axes[0].axvline(x=mejor_n, color="#27ae60", linestyle=":", linewidth=1.5,
                alpha=0.9, label=f"Mejor por test ({mejor_n})")
axes[0].set_title("F1-Score (clase 1) vs n_estimators", fontsize=12, fontweight="bold")
axes[0].set_xlabel("n_estimators")
axes[0].set_ylabel("F1-Score")
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- Panel derecho: gap ---
axes[1].plot(df_curva.n_estimators, df_curva.gap, "o-", color="#f39c12")
axes[1].axhline(y=0.05, color="#e74c3c", linestyle="--", linewidth=1.2, label="Umbral 5pp")
axes[1].fill_between(
    df_curva.n_estimators, 0, df_curva.gap,
    where=df_curva.gap > 0.05,
    alpha=0.15, color="#e74c3c", label="Zona de overfitting"
)
axes[1].set_title("Gap (train − test) vs n_estimators", fontsize=12, fontweight="bold")
axes[1].set_xlabel("n_estimators")
axes[1].set_ylabel("Gap de F1")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(
    "Efecto del número de árboles sobre el overfitting",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 9. Búsqueda de hiperparámetros con GridSearchCV

Se realiza una búsqueda sistemática sobre las dos palancas principales de complejidad del modelo:
- `n_estimators` — número de árboles
- `max_depth` — profundidad máxima de cada árbol

Se usa **validación cruzada de 3 folds** sobre el conjunto de entrenamiento,
optimizando directamente el F1-Score de la clase 1 (la métrica del TP).

> El conjunto de test **no participa** en la búsqueda. Solo se usa para la evaluación final.

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth":    [3, 4, 5, 6],
}

n_combos = len(param_grid["n_estimators"]) * len(param_grid["max_depth"])
print(f"Grid: {param_grid}")
print(f"Combinaciones: {n_combos} × 3 folds = {n_combos * 3} entrenamientos")
print("Ejecutando GridSearchCV (puede tardar 2-4 min)...\n")

base = XGBClassifier(
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    verbosity=0
)

grid_search = GridSearchCV(
    estimator=base,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nMejores parámetros: {grid_search.best_params_}")
print(f"Mejor F1-CV (3 folds): {grid_search.best_score_:.4f}")

In [ ]:
# Tabla resumen de resultados del grid
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = ["param_n_estimators", "param_max_depth",
        "mean_test_score", "std_test_score", "rank_test_score"]

print("Top 10 combinaciones:")
print(
    cv_results[cols]
    .sort_values("rank_test_score")
    .head(10)
    .rename(columns={
        "param_n_estimators": "n_estimators",
        "param_max_depth":    "max_depth",
        "mean_test_score":    "F1-CV (media)",
        "std_test_score":     "std",
        "rank_test_score":    "rank"
    })
    .to_string(index=False)
)

In [ ]:
# Heatmap de resultados: max_depth × n_estimators
pivot = cv_results.pivot_table(
    values="mean_test_score",
    index="param_max_depth",
    columns="param_n_estimators"
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    pivot, annot=True, fmt=".4f",
    cmap="YlGn", linewidths=0.5,
    ax=ax, annot_kws={"size": 11}
)

best_n = grid_search.best_params_["n_estimators"]
best_d = grid_search.best_params_["max_depth"]

# Marcar la celda ganadora
col_idx = list(pivot.columns).index(best_n)
row_idx = list(pivot.index).index(best_d)
ax.add_patch(plt.Rectangle((col_idx, row_idx), 1, 1,
                             fill=False, edgecolor="#e74c3c", lw=3))

ax.set_title(
    f"F1-Score CV (media 3 folds) por combinación de hiperparámetros\n"
    f"Recuadro rojo = mejor combinación (n={best_n}, depth={best_d})",
    fontsize=12, fontweight="bold"
)
ax.set_xlabel("n_estimators", fontsize=11)
ax.set_ylabel("max_depth", fontsize=11)
plt.tight_layout()
plt.show()

## 10. Entrenamiento y evaluación del modelo optimizado

Se entrena el modelo final con los mejores hiperparámetros encontrados por GridSearchCV.
Se compara directamente contra el baseline para cuantificar la mejora.

In [ ]:
best_params = grid_search.best_params_
print(f"Parámetros del modelo optimizado: {best_params}")

model_opt = XGBClassifier(
    **best_params,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    verbosity=0
)
model_opt.fit(X_train, y_train)

y_train_pred_opt = model_opt.predict(X_train)
y_test_pred_opt  = model_opt.predict(X_test)

f1_train_opt = f1_score(y_train, y_train_pred_opt)
f1_test_opt  = f1_score(y_test,  y_test_pred_opt)
gap_opt      = f1_train_opt - f1_test_opt

print("\nModelo optimizado entrenado.")

In [ ]:
print("=" * 50)
print("  REPORTE — MODELO OPTIMIZADO (TEST)")
print("=" * 50)
print(classification_report(
    y_test, y_test_pred_opt,
    target_names=["No fumador (0)", "Fumador (1)"]
))

In [ ]:
# --- Tabla comparativa ---
print(f"{'Modelo':<22} {'F1-Train':>9}  {'F1-Test':>8}  {'Gap':>8}")
print("-" * 52)
print(f"{'Baseline':22} {f1_train:>9.4f}  {f1_test:>8.4f}  {f1_train - f1_test:>+8.4f}")
print(f"{'Optimizado':22} {f1_train_opt:>9.4f}  {f1_test_opt:>8.4f}  {gap_opt:>+8.4f}")
print()

mejora = f1_test_opt - f1_test
if mejora > 0:
    print(f"✓  Mejora en F1-test: +{mejora:.4f} ({mejora*100:.2f} pp)")
else:
    print(f"–  Sin mejora en F1-test: {mejora:.4f}")

reduccion_gap = (f1_train - f1_test) - gap_opt
print(f"✓  Reducción de gap: {reduccion_gap:.4f} ({reduccion_gap*100:.2f} pp)")

# --- Gráfico comparativo ---
categorias = ["Precision\n(clase 1)", "Recall\n(clase 1)", "F1-Score\n(clase 1)"]

val_base = [
    precision_score(y_test, y_test_pred),
    recall_score(y_test, y_test_pred),
    f1_test
]
val_opt = [
    precision_score(y_test, y_test_pred_opt),
    recall_score(y_test, y_test_pred_opt),
    f1_test_opt
]

x    = np.arange(len(categorias))
ancho = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - ancho/2, val_base, ancho, label="Baseline",   color="#95a5a6", edgecolor="white")
bars2 = ax.bar(x + ancho/2, val_opt,  ancho, label="Optimizado", color="#2ecc71", edgecolor="white")

for bar in list(bars1) + list(bars2):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{bar.get_height():.4f}",
        ha="center", va="bottom", fontsize=9, fontweight="bold"
    )

ax.set_xticks(x)
ax.set_xticklabels(categorias, fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title(
    "Baseline vs Optimizado — Métricas sobre el conjunto de TEST (clase 1 = fumador)",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión del modelo optimizado
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (y_true, y_pred, titulo) in zip(axes, [
    (y_train, y_train_pred_opt, "Train"),
    (y_test,  y_test_pred_opt,  "Test")
]):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No fumador (0)", "Fumador (1)"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Greens")
    ax.set_title(f"Matriz de Confusión — {titulo} (optimizado)",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicción", fontsize=10)
    ax.set_ylabel("Real", fontsize=10)

plt.tight_layout()
plt.show()

## 11. Guardado del modelo optimizado

In [ ]:
out_opt = PATH_MODELS / "xgboost_optimized.joblib"
joblib.dump(model_opt, out_opt)
print(f"✓ Modelo optimizado guardado: {out_opt}  ({out_opt.stat().st_size/1024:.1f} KB)")

# Guardar también los mejores parámetros para referencia
import json
best_params_full = {
    **best_params,
    "learning_rate":    0.1,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "random_state":     RANDOM_STATE,
    "f1_test":          round(f1_test_opt, 4),
    "f1_train":         round(f1_train_opt, 4),
    "gap":              round(gap_opt, 4)
}
with open(PATH_MODELS / "best_params.json", "w") as f:
    json.dump(best_params_full, f, indent=2)
print(f"✓ Parámetros guardados: {PATH_MODELS / 'best_params.json'}")
print(json.dumps(best_params_full, indent=2))

## 12. Resumen comparativo

### Resultados

| Modelo | F1-Train | F1-Test | Gap | n_estimators | max_depth |
|---|---|---|---|---|---|
| Baseline | 0.7511 | 0.6947 | +0.0564 | 100 | 6 |
| Optimizado (GridSearchCV) | 0.7511 | 0.6947 | +0.0564 | **100** | **6** |

### Hallazgo clave: el baseline ya era el punto óptimo

La GridSearchCV (cv=3) confirmó que los parámetros del baseline son los mejores dentro del grid explorado (`n_estimators` × `max_depth`).

La **curva de n_estimators** revela el comportamiento completo:

| n_estimators | F1-train | F1-test | Gap |
|---|---|---|---|
| 10 | 0.6273 | 0.6110 | +0.016 — **underfitting** |
| 50 | 0.7264 | 0.6895 | +0.037 |
| **100** | **0.7511** | **0.6947** | **+0.056** ← baseline |
| 150 | 0.7717 | 0.7008 | +0.071 |
| 300 | 0.8271 | 0.7013 | +0.126 — **overfitting severo** |

Con n=150 el F1-test sube marginalmente (+0.006) pero el gap crece +0.015.
El GridSearchCV eligió n=100 por ser más robusto en validación cruzada (F1-CV=0.6937).

**Conclusión:** el gap de ~5.6 pp no se debe a una elección incorrecta de hiperparámetros sino a la naturaleza del problema. Reducir `max_depth` o `n_estimators` baja el F1-test antes de reducir el gap. Para mejorar la generalización habría que explorar regularizadores adicionales (`min_child_weight`, `gamma`, `reg_alpha`) o técnicas de feature engineering.

### Decisiones de optimización

| Paso | Herramienta | Parámetros explorados |
|---|---|---|
| Análisis de overfitting | Curva manual | `n_estimators` de 10 a 300 |
| Búsqueda sistemática | `GridSearchCV` (cv=3) | `n_estimators` [50,100,150,200] × `max_depth` [3,4,5,6] |
| Modelo final | Parámetros del baseline, confirmados óptimos | — |

### Modelos guardados en `models/`

| Archivo | Contenido |
|---|---|
| `xgboost_baseline.joblib` | Modelo baseline (n=100, depth=6) |
| `xgboost_optimized.joblib` | Modelo confirmado óptimo por GridSearchCV (mismos parámetros) |
| `best_params.json` | Hiperparámetros y métricas del modelo seleccionado |

### Próximo paso

El notebook `06_prediccion.ipynb` carga `xgboost_optimized.joblib` y genera las predicciones finales sobre el dataset de entrega (`smoking_predict_processed.csv`).

---
## 13. Investigación del overfitting: análisis por subgrupo de género

Con el gap de 5.6 pp confirmado como persistente después de la optimización, se investigaron dos hipótesis:

1. **¿Hay data drift?** — ¿El dataset de entrega tiene una distribución diferente a la de entrenamiento? Si `gender` u otras variables se distribuyen de forma distinta, el modelo podría errar más en producción.
2. **¿El error está concentrado en algún subgrupo?** — Analizar el F1-Score separando hombres y mujeres para identificar dónde se origina el gap.

> Esta investigación parte de la observación del EDA: `gender` es la variable más predictiva (importancia 0.808), lo que hace que su distribución y comportamiento por subgrupo sean críticos para entender el modelo.

### 13.1. ¿Hay data drift entre el dataset de entrenamiento y el de entrega?

In [ ]:
df_predict = pd.read_csv(PATH_PROC / "smoking_predict_processed.csv")

features_clave = ["age", "hemoglobin", "Gtp", "tartar", "weight(kg)",
                  "height(cm)", "systolic", "triglyceride", "Cholesterol"]

print(f"{'Feature':<20} {'Train (media)':>14}  {'Predict (media)':>16}  {'Diferencia':>11}")
print("-" * 65)
for col in ["gender"] + features_clave:
    t_mean = X[col].mean()
    p_mean = df_predict[col].mean()
    diff   = abs(t_mean - p_mean)
    print(f"{col:<20} {t_mean:>14.4f}  {p_mean:>16.4f}  {diff:>11.4f}")

print()
print("=== Distribución de gender ===")
print(f"Train   → Hombres (1): {X['gender'].mean()*100:.1f}%  |  Mujeres (0): {(1-X['gender'].mean())*100:.1f}%")
print(f"Predict → Hombres (1): {df_predict['gender'].mean()*100:.1f}%  |  Mujeres (0): {(1-df_predict['gender'].mean())*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ["gender", "age", "hemoglobin"]):
    if col == "gender":
        labels = ["Mujer (0)", "Hombre (1)"]
        vals_tr = [( X[col] == 0).mean(), ( X[col] == 1).mean()]
        vals_pr = [(df_predict[col] == 0).mean(), (df_predict[col] == 1).mean()]
        x_pos = np.arange(len(labels))
        ax.bar(x_pos - 0.18, vals_tr, 0.35, label="Train",   color="#3498db", alpha=0.85)
        ax.bar(x_pos + 0.18, vals_pr, 0.35, label="Predict", color="#e67e22", alpha=0.85)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Proporción")
    else:
        ax.hist(X[col],          bins=30, alpha=0.6, color="#3498db", label="Train",   density=True)
        ax.hist(df_predict[col], bins=30, alpha=0.6, color="#e67e22", label="Predict", density=True)
        ax.set_xlabel(col)
        ax.set_ylabel("Densidad")

    ax.set_title(f"Distribución: {col}", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle(
    "Comparación de distribuciones: Train vs Dataset de entrega",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

### 13.2. Análisis de errores por subgrupo de género

Dado que `gender` domina el modelo, se analiza el F1-Score y la matriz de confusión de forma separada para hombres y mujeres, tanto en train como en test.

In [ ]:
# Predicciones del modelo optimizado sobre train y test (ya disponibles)
test_analysis  = X_test.copy()
test_analysis["smoking_real"] = y_test.values
test_analysis["pred"]         = y_test_pred_opt

train_analysis  = X_train.copy()
train_analysis["smoking_real"] = y_train.values
train_analysis["pred"]         = y_train_pred_opt

print(f"{'Subgrupo':<10} {'Set':<6} {'Total':>7}  {'Fuman (real)':>13}  {'% fuman':>8}  {'Predice fuma':>13}  {'F1-Score':>9}")
print("-" * 75)

for g_val, g_name in [(0, "Mujer"), (1, "Hombre")]:
    for df_sub, set_name in [(train_analysis, "Train"), (test_analysis, "Test")]:
        sub = df_sub[df_sub["gender"] == g_val]
        n_real = (sub["smoking_real"] == 1).sum()
        n_pred = (sub["pred"] == 1).sum()
        pct    = n_real / len(sub) * 100
        f1     = f1_score(sub["smoking_real"], sub["pred"], zero_division=0)
        print(f"{g_name:<10} {set_name:<6} {len(sub):>7}  {n_real:>13}  {pct:>7.1f}%  {n_pred:>13}  {f1:>9.4f}")
    print()

In [ ]:
# Matrices de confusión por subgrupo — solo en el conjunto de TEST
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (g_val, g_name) in zip(axes, [(0, "Mujeres"), (1, "Hombres")]):
    sub = test_analysis[test_analysis["gender"] == g_val]
    cm  = confusion_matrix(sub["smoking_real"], sub["pred"])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No fumador (0)", "Fumador (1)"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")

    n_fuman = (sub["smoking_real"] == 1).sum()
    pct     = n_fuman / len(sub) * 100
    f1      = f1_score(sub["smoking_real"], sub["pred"], zero_division=0)

    ax.set_title(
        f"{g_name} — TEST\n"
        f"n={len(sub)}  |  Fuman: {n_fuman} ({pct:.1f}%)  |  F1={f1:.4f}",
        fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("Predicción", fontsize=10)
    ax.set_ylabel("Real", fontsize=10)

plt.suptitle(
    "Matrices de confusión por subgrupo de género (modelo optimizado — TEST)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico del gap F1 por subgrupo: train vs test
subgrupos = ["Mujeres", "Hombres", "Global"]
f1_tr_vals, f1_te_vals = [], []

for g_val in [0, 1]:
    for df_sub, lst in [(train_analysis, f1_tr_vals), (test_analysis, f1_te_vals)]:
        sub = df_sub[df_sub["gender"] == g_val]
        lst.append(f1_score(sub["smoking_real"], sub["pred"], zero_division=0))

f1_tr_vals.append(f1_score(y_train, y_train_pred_opt))
f1_te_vals.append(f1_score(y_test,  y_test_pred_opt))

x_pos = np.arange(len(subgrupos))
ancho = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_tr = ax.bar(x_pos - ancho/2, f1_tr_vals, ancho, label="Train", color="#3498db", alpha=0.85)
bars_te = ax.bar(x_pos + ancho/2, f1_te_vals, ancho, label="Test",  color="#e74c3c", alpha=0.85)

for bars in [bars_tr, bars_te]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.007,
            f"{bar.get_height():.4f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold"
        )

for i, (tr, te) in enumerate(zip(f1_tr_vals, f1_te_vals)):
    gap = tr - te
    ax.annotate(
        f"gap\n{gap:+.4f}",
        xy=(x_pos[i], max(tr, te) + 0.06),
        ha="center", fontsize=9, color="#7f8c8d",
        fontweight="bold"
    )

ax.set_xticks(x_pos)
ax.set_xticklabels(subgrupos, fontsize=12)
ax.set_ylabel("F1-Score (clase 1 = fumador)", fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title(
    "Gap de overfitting (train − test) por subgrupo de género",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{'Subgrupo':<10} {'F1-Train':>10}  {'F1-Test':>9}  {'Gap':>8}")
print("-" * 42)
for nombre, tr, te in zip(subgrupos, f1_tr_vals, f1_te_vals):
    print(f"{nombre:<10} {tr:>10.4f}  {te:>9.4f}  {tr-te:>+8.4f}")

### 13.3. Conclusiones de la investigación

#### Hipótesis 1 — Data drift: **DESCARTADA**

Las distribuciones del dataset de entrenamiento y del dataset de entrega son prácticamente idénticas:
- `gender`: 63.5% hombres en train vs 63.5% en predict
- Todas las features numéricas tienen diferencias de medias menores a 0.15 unidades

No hay drift. El modelo debería rendir de forma similar sobre el dataset de entrega.

---

#### Hipótesis 2 — Origen del gap: **CONFIRMADA**

| Subgrupo | F1-Train | F1-Test | Gap |
|---|---|---|---|
| **Mujeres** | ~0.13 | ~0.01 | **~0.12 ← gap dominante** |
| Hombres | ~0.77 | ~0.71 | ~0.06 |
| **Global** | **0.7511** | **0.6947** | **0.0564** |

El gap global de 5.6 pp está **dominado por el colapso del modelo en el subgrupo de mujeres**.

#### ¿Por qué el modelo falla con las mujeres fumadoras?

Es un problema de **desbalance de clase a nivel de subgrupo**:
- Solo el **4.0% de las mujeres** en el dataset son fumadoras (~144 de 3,625 en test)
- El modelo aprende que lo más "seguro" es predecir `0` (no fumadora) para casi todas las mujeres
- Esta regla es 96% precisa en accuracy, pero resulta en un F1 ≈ 0 para la clase 1 dentro del subgrupo

En test detecta apenas **1 mujer fumadora de 144** (144 falsos negativos, 3 falsos positivos).

#### ¿Cómo podría mejorarse?

El problema no se soluciona con más árboles ni mayor profundidad. Las alternativas son:

| Técnica | Mecanismo | Trade-off |
|---|---|---|
| `scale_pos_weight` | Da más peso a la clase positiva durante el entrenamiento | Puede generar más FP en hombres |
| `sample_weight` | Sobrepesa a las mujeres fumadoras específicamente | Requiere calibración manual |
| Ajuste de threshold | Bajar el umbral de 0.5 a 0.3 para predecir más fumadores | Reduce precisión global |

Para el alcance de este TP, el modelo actual con F1-test = 0.6947 es la solución adoptada.